In [ ]:
!nvidia-smi -L
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1 && apt-get install -y -qq nodejs >/dev/null 2>&1
!apt-get install -y -qq libnss3 libdbus-1-3 libatk1.0-0 libgbm-dev libasound2 libxrandr2 libxkbcommon-dev libxfixes3 libxcomposite1 libxdamage1 libatk-bridge2.0-0 libpango-1.0-0 libcairo2 libcups2 >/dev/null 2>&1
!rm -rf /content/sc && git clone -q --depth 1 https://github.com/andyheree-design/stadt-cloud /content/sc
%cd /content/sc
!npm ci --no-audit --no-fund >/dev/null 2>&1; node -v
!npx remotion browser ensure --chrome-mode=chrome-for-testing >/dev/null 2>&1
!npx remotion bundle cloud/index.ts --out-dir=build >/dev/null 2>&1; ls build | head -3

In [ ]:
import subprocess, time, os
from google.colab import files
A, B, K = 1890, 3734, 100
GL = os.environ.get('GL', 'angle-egl')
os.makedirs('out', exist_ok=True)
for a in range(A, B + 1, K):
    b = min(a + K - 1, B); out = f'out/part_{a:06d}.mp4'; t = time.time()
    r = subprocess.run(['npx','remotion','render','build','StadtE02',out,f'--frames={a}-{b}',f'--gl={GL}',
        '--chrome-mode=chrome-for-testing','--concurrency=4','--muted','--codec=h264','--crf=16',
        '--pixel-format=yuv420p','--timeout=120000','--log=error'], capture_output=True, text=True)
    dt = time.time() - t
    print(f'{a}-{b} exit={r.returncode} {dt:.0f}s = {dt/(b-a+1):.2f} s/kl.', r.stderr[-400:] if r.returncode else '', flush=True)
    if r.returncode: break
    files.download(out)